In [47]:
import pandas as pd

#resturants
df_rests = pd.read_csv(
    "../Project/team_project_dataset/restaurants_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_rests.columns = df_rests.iloc[0]
df_rests = df_rests[1:]
df_rests.reset_index(drop=True)

#test
df_test = pd.read_csv(
    "../Project/team_project_dataset/test_reviews_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_test.columns = df_test.iloc[0]
df_test = df_test[1:]
df_test.reset_index(drop=True)

#train
df_train = pd.read_csv(
    "../Project/team_project_dataset/train_reviews_santa_barbara.csv",
    sep=",",
    encoding="latin-1",
    header=None
)

df_train.columns = df_train.iloc[0]
df_train = df_train[1:]
df_train.reset_index(drop=True)

df_rests

,business_id,name,address,city,state,postal_code,categories,latitude,longitude,attributes
1,IDtLPgUrqorrpqSLdfMhZQ,Helena Avenue Bakery,"131 Anacapa St, Ste C",Santa Barbara,CA,93101,"food, restaurants, salad, coffee & tea, breakf...",34.4144445,-119.6906718,"{'RestaurantsTakeOut': 'True', 'NoiseLevel': ""..."
2,SZU9c8V2GuREDN5KgyHFJw,Santa Barbara Shellfish Company,230 Stearns Wharf,Santa Barbara,CA,93101,"live/raw food, restaurants, seafood, beer bar,...",34.4087147,-119.6850187,"{'OutdoorSeating': 'True', 'RestaurantsAttire'..."
3,ifjluUv4VASwmFqEp8cWlQ,Marty's Pizza,2733 De La Vina St,Santa Barbara,CA,93105,"pizza, restaurants",34.4362361,-119.7261474,"{'Alcohol': ""u'none'"", 'BusinessAcceptsCreditC..."
4,UFpCraqzFBAhtZqmxmiWsA,Cat Therapy,"1213 State St, Ste L",Santa Barbara,CA,93101,"themed cafes, cafes, pets, arts & entertainmen...",34.4233018995,-119.7054707479,"{'WheelchairAccessible': 'True', 'WiFi': ""u'fr..."
5,Hqz96v1ymucUKNzIWfEKXw,Subway,"1936 State St, Ste B",Santa Barbara,CA,93101,"restaurants, delis, sandwiches, fast food",34.430822,-119.714156,"{'Alcohol': ""u'none'"", 'Caters': 'True', 'Bike..."
...,...,...,...,...,...,...,...,...,...,...
763,Hlx8S2GLF7hMuIKx4sU-gg,Cesar's Place,712 N Milpas St,Santa Barbara,CA,93103,"mexican, restaurants, fish & chips",34.4285995,-119.6882227,"{'BYOBCorkage': ""'no'"", 'RestaurantsReservatio..."
764,bVaRZDHkWdsHuARGxxpREw,Sushi Bar 29,1134 Chapala St,Santa Barbara,CA,93101,"japanese, restaurants, noodles, sushi bars",34.422291,-119.705339,"{'WiFi': ""u'no'"", 'RestaurantsTableService': '..."
765,izSgTrqebu8bN8ONOCs6cQ,Oat Bakery,5 W Haley St,Santa Barbara,CA,93101,"bakeries, vegan, specialty food, food delivery...",34.4165478074,-119.6956258296,"{'Alcohol': ""u'none'"", 'HasTV': 'False', 'Bike..."
766,TSwMwVq5GtQh5LW2t32uGA,Woody's Roundup Bar & Grill,"Earl Warren Showgrounds, 3400 Calle Real",Santa Barbara,CA,93105,"barbeque, bars, restaurants, nightlife",34.4308945309,-119.7356450558,"{'Caters': 'True', 'BusinessParking': ""{'garag..."


In [48]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


#FIRST I WANT TO EMBED THE ITEMS (THE RESTAURANTS) AND GET VECTORS FOR EACH
df_rests = df_rests.reset_index(drop=True)
embedder_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder_model.encode(df_rests["categories"])
df_rests["category_embeddings"] = list(embeddings)

In [50]:
#on top of the categorical embeddings, I want to add to each embedding a few values connoting attribute values

#first clean values for the attribute column
import ast
def str_to_dict(s):
    try:
        return ast.literal_eval(s)
    except: #need this or else it won't run
        return {}
df_rests['attributes_dict'] = df_rests['attributes'].apply(str_to_dict)

def clean_values(d):
    return {k: (v.strip("'\"") if isinstance(v, str) else v) for k, v in d.items()}
df_rests['attributes_dict'] = df_rests['attributes_dict'].apply(clean_values)

#then use RestaurantsPriceLevel2, GoodForKids, and RestaurantsAttire; these are all good indicators because people feel strongly about prices, about their families, and about "comfortability" of a restaurant. A person who wants to eat cheap will really appreciate a cheap recommendation, a person who has kids will really appreciate a restaurant that caters to them, and a person who likes to be comfortable/homey will really appreciate a restaurant recommendation that feels like home.
df_rests['good_for_kids'] = 0
maskgfk = df_rests['attributes_dict'].apply(lambda d: d.get('GoodForKids') == 'True')
df_rests.loc[maskgfk, 'good_for_kids'] = 1

df_rests['rest_p_level_1'] = 0
df_rests['rest_p_level_2'] = 0
mask1 = df_rests['attributes_dict'].apply(lambda d: d.get('RestaurantsPriceRange2') == '1')
mask2 = df_rests['attributes_dict'].apply(lambda d: d.get('RestaurantsPriceRange2') == '2')

df_rests.loc[mask1, 'rest_p_level_1'] = 1
df_rests.loc[mask2, 'rest_p_level_2'] = 1

df_rests['casual'] = 0
maskcas = df_rests['attributes_dict'].apply(lambda d: d.get('RestaurantsAttire') == 'casual')
df_rests.loc[maskcas, 'casual'] = 1

#then, you want these values to enter the category embedding to make one big embedding, so you can calculate similarities of single vectors.
import numpy as np

df_rests["new_embedding"] = df_rests.apply(lambda row: np.concatenate([row["category_embeddings"],
        [row["good_for_kids"], row["rest_p_level_1"], row["rest_p_level_2"], row['casual']]]), axis=1)


In [51]:
#NOW WE NEED USER PROFILES FOR EACH USER IN THE df_train

#we want to include ranking, although we are not including review text
#to implement this, we want to get a rating bias for each restaurant (whether it was rated more positively or negatively than the user average) and then use that to weight the restaurant vectors in the user profile (so that the recommendations we get are more similar to the restaurants the user liked more)
#first get the mean rating for each user
df_train['stars'] = pd.to_numeric(df_train['stars'])
user_means = df_train.groupby("user_id")["stars"].mean()
df_train = df_train.merge(user_means.rename("user_mean"), on="user_id")

In [52]:
#now get the rating bias for each user
df_train["adjusted_rating"] = df_train["stars"] - df_train["user_mean"]

In [53]:
#put everything together
merge = df_train.merge(df_rests, on="business_id")

In [57]:
#now we want to get the user profiles
merge = merge.set_index("business_id")
user_profiles = {}
for user, group in df_train.groupby("user_id"):
    vectors = merge.loc[group["business_id"], "new_embedding"]
    user_profiles[user] = np.vstack(vectors).mean(axis=0)


In [68]:
#get recommendations
from sklearn.metrics.pairwise import cosine_similarity

def recommend_restaurants(user_id, user_embeddings, restaurants_df, k):
    user_vec = user_embeddings[user_id].reshape(1, -1)
    restaurant_matrix = np.vstack(restaurants_df['new_embedding'].values)

    similarities = cosine_similarity(user_vec, restaurant_matrix)[0]
    top_k_idx = np.argsort(similarities)[-k:][::-1]

    results = restaurants_df[['business_id', 'name']].iloc[top_k_idx].copy()
    results['similarity'] = similarities[top_k_idx]

    return results

In [70]:
#test
ay = recommend_restaurants('-0EcgtUXe1rzrkmdih_tYg', user_profiles, df_rests, 10)

In [78]:
#now we can do accuracy metrics. first is NDCG@k
def ndcg_at_k(recommended_ids, true_id, k):
    try:
        rank = recommended_ids.index(true_id) + 1
    except ValueError:
        return 0.0

    return 1 / np.log2(rank + 1)

def evaluate_ndcg(user_embeddings, restaurants_df, test_df, k):
    scores = []

    for _, row in test_df.iterrows():
        user_id = row['user_id']
        true_business_id = row['business_id']

        recs = recommend_restaurants(user_id,user_embeddings,restaurants_df,k)
        recommended_ids = recs['business_id'].tolist()
        score = ndcg_at_k(recommended_ids, true_business_id, k)
        scores.append(score)

    return np.mean(scores)

#then hit@k
def hit_at_k(recommended_ids, true_id):
    return int(true_id in recommended_ids)

import numpy as np

def evaluate_hit_at_k(user_embeddings, restaurants_df, test_df, k):
    hits = []
    for _, row in test_df.iterrows():
        user_id = row['user_id']
        true_business_id = row['business_id']
        recs = recommend_restaurants(user_id, user_embeddings, restaurants_df, k)

        recommended_ids = recs['business_id'].tolist()

        hit = hit_at_k(recommended_ids, true_business_id)
        hits.append(hit)

    return np.mean(hits)

In [80]:
for k in [10, 20, 30]:
    print(f"Hit@{k}:", evaluate_hit_at_k(user_profiles, df_rests, df_test, k), f"\t\tNDCG@{k}:", evaluate_ndcg(user_profiles, df_rests, df_test, k))

Hit@10: 0.026452822328681524 		NDCG@10: 0.010523213724964196
Hit@20: 0.05977921266402833 		NDCG@20: 0.01880339486892418
Hit@30: 0.08539887523432618 		NDCG@30: 0.024271393749984223
